# 업로드 확인

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w6" / "day01"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [2]:
from app.rag import chunker, parser

p = next(SAMPLES.glob('DOC-FI-009_*_v1.4.docx'))
doc = parser.parse(p)
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}')
drafts = chunker.chunk(doc)
print(chunker.summarize(drafts))
for i, d in enumerate(drafts):
    if d.kind == '표':
        print(f'  [표] ord={i}  locator={d.locator}  글자 수={len(d.text)}')
print('★ 이 drafts 리스트는 이 셀이 끝나면 사라집니다. 담을 자리가 없습니다.')

블록 81 · 표 2 · 쪽 1
조항 단위 17 + 표 단위 2 = 19 청크
  [표] ord=1  locator=표1  글자 수=264
  [표] ord=18  locator=표2  글자 수=283
★ 이 drafts 리스트는 이 셀이 끝나면 사라집니다. 담을 자리가 없습니다.


In [3]:
# 노트북 셀에서 install 하는 방법 
%pip install -U ipywidgets jupyter ipykernel

Note: you may need to restart the kernel to use updated packages.


In [4]:
import importlib
import app.models

importlib.reload(app.models)

print('Base.metadata 에 등록된 표:', sorted(app.models.Base.metadata.tables))

Base.metadata 에 등록된 표: ['chunks', 'departments', 'document_versions', 'documents', 'run_steps', 'runs', 'usage_logs', 'users']


In [5]:
import importlib
from app.services import document_service

UPLOADED = "uploads/DOC-FI-009_v1.4.docx" 

importlib.reload(document_service)
print(document_service.ingest_document(doc_id="DOC-FI-009", version="v1.4", path=UPLOADED))

{'chunks': 19, 'summary': '조항 단위 17 + 표 단위 2 = 19 청크'}


In [6]:
import time
from app.rag import chunker, parser

PATTERNS = ['DOC-FI-009_*_v1.4.docx', 'DOC-FI-009_*_v1.4.pdf', 'DOC-HR-014_*_v2.0.docx', 'DOC-HR-014_*_v2.0.pdf']

for pattern in PATTERNS:
    path = next(SAMPLES.glob(pattern))
    started = time.perf_counter()
    parsed = parser.parse(path)
    drafts = chunker.chunk(parsed)
    elapsed = time.perf_counter() - started
    print(f'{path.name:<34} {len(drafts):>3}청크   {elapsed:.3f}초')
TIMEOUT = 10.0
print()
print(f'TIMEOUT = {TIMEOUT} 초 — 우리 재료는 전부 그 아래입니다.')


DOC-FI-009_법인카드_사용_지침_v1.4.docx     19청크   0.012초
DOC-FI-009_법인카드_사용_지침_v1.4.pdf      17청크   0.080초
DOC-HR-014_국내출장_여비_규정_v2.0.docx     53청크   0.014초
DOC-HR-014_국내출장_여비_규정_v2.0.pdf      45청크   0.063초

TIMEOUT = 10.0 초 — 우리 재료는 전부 그 아래입니다.
